In [1]:
import os

In [2]:
main = r"C:\Users\shash\OneDrive - Nord Anglia Education\IB"

import re

papers = []

In [3]:
for subject in os.listdir(main):
    subject_path = os.path.join(main, subject)
    if "Past Papers" in os.listdir(subject_path):
        past_papers_path = os.path.join(subject_path, "Past Papers")
        for root, dirs, files in os.walk(past_papers_path):
            for file in files:
                if file.endswith(".pdf"):
                    file_path = os.path.join(root, file)
                    file_props = {}
                    file_props["subject"] = subject
                    file_props["level"] = re.findall(r"HL|SL", file_path)[0]
                    file_props["year"] = int(re.findall(r"\d{4}", file_path)[0])
                    file_props["session"] = re.findall(r"May|November", file_path)[0]
                    file_props["paper"] = int(re.findall(r"paper_\d", file_path)[0][-1]) if len(re.findall(r"paper_\d", file_path)) > 0 else re.findall(r"\\\w+__[HL|SL]", file_path)[0]
                    file_props["Timezone"] = 1 if len(re.findall(r"TZ_*\d", file_path)) < 1 else re.findall(r"TZ_*\d", file_path)[0][-1]
                    file_props["language"] = re.findall(r"English|French|Spanish", file_path)[0] if len(re.findall(r"English|French|Spanish", file_path)) > 0 else "English"
                    file_props["type"] = re.findall(r"markscheme", file_path)[0] if len(re.findall(r"markscheme", file_path)) > 0 else "Question Paper"
                    file_props["file_path"] = file_path
                    papers.append(file_props)


In [9]:
import pyodbc
conn_str = (
    r'DRIVER={Microsoft Access Driver (*.mdb, *.accdb)};'
    r'DBQ=C:\Users\shash\Desktop\IB_Papers.accdb;'
)
conn = pyodbc.connect(conn_str)
cursor = conn.cursor()

In [11]:
for paper in papers:
    cursor.execute("INSERT INTO Papers ([Subject], [Level], [Annum], [Session], [Paper], [Timezone], [Type],[Path], [Language]) VALUES (?, ?, ?, ?,?,?,?,?,?);", 
                   paper["subject"], paper["level"], paper["year"], paper["session"], paper["paper"], paper["Timezone"], paper['type'], paper["file_path"], paper["language"])
conn.commit()
cursor.close()
conn.close()